In [2]:
from pyspark.sql import SparkSession

# 1. Initialize SparkSession
spark = SparkSession.builder \
    .appName("PySpark-Practice") \
    .master("local[*]") \
    .getOrCreate()

# 2. Create sample data
data = [("Alice", 34), ("Bob", 45), ("Charlie", 29)]
columns = ["Name", "Age"]

# 3. Create DataFrame
df = spark.createDataFrame(data, columns)

# 4. Show the DataFrame
df.show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/28 14:28:44 WARN Utils: Your hostname, codespaces-d0a7aa, resolves to a loopback address: 127.0.0.1; using 10.0.3.77 instead (on interface eth0)
26/10/28 14:28:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/python/3.14.2/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/28 14:28:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 34|
|    Bob| 45|
|Charlie| 29|
+-------+---+



#### Reading data

In [ ]:
# Read fact table

df_2020 = spark.read.csv(
    "/workspaces/PySpark-Recap/Sample_Excel_Files/AdventureWorks Sales Data 2020.csv", 
    header=True, 
    inferSchema=True
)

df_2021 = spark.read.csv(
    "/workspaces/PySpark-Recap/Sample_Excel_Files/AdventureWorks Sales Data 2021.csv", 
    header=True, 
    inferSchema=True
)

df_2022 = spark.read.csv(
    "/workspaces/PySpark-Recap/Sample_Excel_Files/AdventureWorks Sales Data 2022.csv", 
    header=True, 
    inferSchema=True
)

df = df_2020.unionByName(df_2021).unionByName(df_2022)

df.show()


+----------+----------+-----------+----------+-----------+------------+-------------+-------------+
| OrderDate| StockDate|OrderNumber|ProductKey|CustomerKey|TerritoryKey|OrderLineItem|OrderQuantity|
+----------+----------+-----------+----------+-----------+------------+-------------+-------------+
|2020-01-01|2019-09-21|    SO45080|       332|      14657|           1|            1|            1|
|2020-01-01|2019-12-05|    SO45079|       312|      29255|           4|            1|            1|
|2020-01-01|2019-10-29|    SO45082|       350|      11455|           9|            1|            1|
|2020-01-01|2019-11-16|    SO45081|       338|      26782|           6|            1|            1|
|2020-01-02|2019-12-15|    SO45083|       312|      14947|          10|            1|            1|
|2020-01-02|2019-10-12|    SO45084|       310|      29143|           4|            1|            1|
|2020-01-02|2019-12-18|    SO45086|       314|      18747|           9|            1|            1|


In [6]:
#read lookup table

df_cust = spark.read.csv(
    "/workspaces/PySpark-Recap/Sample_Excel_Files/AdventureWorks Customer Lookup.csv", 
    header=True, 
    inferSchema=True
)

df_cust.show()

+-----------+------+---------+--------+----------+-------------+------+--------------------+------------+-------------+---------------+--------------+---------+
|CustomerKey|Prefix|FirstName|LastName| BirthDate|MaritalStatus|Gender|        EmailAddress|AnnualIncome|TotalChildren| EducationLevel|    Occupation|HomeOwner|
+-----------+------+---------+--------+----------+-------------+------+--------------------+------------+-------------+---------------+--------------+---------+
|      11000|   MR.|      JON|    YANG|1966-04-08|            M|     M|jon24@adventure-w...|       90000|            2|      Bachelors|  Professional|        Y|
|      11001|   MR.|   EUGENE|   HUANG|1965-05-14|            S|     M|eugene10@adventur...|       60000|            3|      Bachelors|  Professional|        N|
|      11002|   MR.|    RUBEN|  TORRES|1965-08-12|            M|     M|ruben35@adventure...|       60000|            3|      Bachelors|  Professional|        Y|
|      11003|   MS.|  CHRISTY|    

In [8]:
df_territory = spark.read.csv(
    "/workspaces/PySpark-Recap/Sample_Excel_Files/AdventureWorks Territory Lookup.csv", 
    header=True, 
    inferSchema=True
)

df_territory.show()

+-----------------+--------------+--------------+-------------+
|SalesTerritoryKey|        Region|       Country|    Continent|
+-----------------+--------------+--------------+-------------+
|                1|     Northwest| United States|North America|
|                2|     Northeast| United States|North America|
|                3|       Central| United States|North America|
|                4|     Southwest| United States|North America|
|                5|     Southeast| United States|North America|
|                6|        Canada|        Canada|North America|
|                7|        France|        France|       Europe|
|                8|       Germany|       Germany|       Europe|
|                9|     Australia|     Australia|      Pacific|
|               10|United Kingdom|United Kingdom|       Europe|
+-----------------+--------------+--------------+-------------+



In [10]:
df_product = spark.read.csv(
    "/workspaces/PySpark-Recap/Sample_Excel_Files/AdventureWorks Product Lookup.csv", 
    header=True, 
    inferSchema=True
)

df_product.show()

+----------+---------------------+----------+--------------------+--------------------+--------------------+------------+-----------+------------+-----------+------------+
|ProductKey|ProductSubcategoryKey|ProductSKU|         ProductName|           ModelName|  ProductDescription|ProductColor|ProductSize|ProductStyle|ProductCost|ProductPrice|
+----------+---------------------+----------+--------------------+--------------------+--------------------+------------+-----------+------------+-----------+------------+
|       214|                   31| HL-U509-R|Sport-100 Helmet,...|           Sport-100|Universal fit, we...|         Red|          0|           0|    13.0863|       34.99|
|       215|                   31|   HL-U509|Sport-100 Helmet,...|           Sport-100|Universal fit, we...|       Black|          0|           0|    12.0278|     33.6442|
|       218|                   23| SO-B909-M|Mountain Bike Soc...| Mountain Bike Socks|Combination of na...|       White|          M|       

In [11]:
df_product_subcat = spark.read.csv(
    "/workspaces/PySpark-Recap/Sample_Excel_Files/AdventureWorks Product Subcategories Lookup.csv", 
    header=True, 
    inferSchema=True
)

df_product_subcat.show()

+---------------------+---------------+------------------+
|ProductSubcategoryKey|SubcategoryName|ProductCategoryKey|
+---------------------+---------------+------------------+
|                    1| Mountain Bikes|                 1|
|                    2|     Road Bikes|                 1|
|                    3|  Touring Bikes|                 1|
|                    4|     Handlebars|                 2|
|                    5|Bottom Brackets|                 2|
|                    6|         Brakes|                 2|
|                    7|         Chains|                 2|
|                    8|      Cranksets|                 2|
|                    9|    Derailleurs|                 2|
|                   10|          Forks|                 2|
|                   11|       Headsets|                 2|
|                   12|Mountain Frames|                 2|
|                   13|         Pedals|                 2|
|                   14|    Road Frames|                 

In [12]:
df_product_cat = spark.read.csv(
    "/workspaces/PySpark-Recap/Sample_Excel_Files/AdventureWorks Product Categories Lookup.csv", 
    header=True, 
    inferSchema=True
)

df_product_cat.show()

+------------------+------------+
|ProductCategoryKey|CategoryName|
+------------------+------------+
|                 1|       Bikes|
|                 2|  Components|
|                 3|    Clothing|
|                 4| Accessories|
+------------------+------------+



#### Practice Questions